# Retail Shopping Assistant

## Notebook 4: Capture Journey Traces

How well a model server handles this agent depends on the requests the agent
makes. Each turn sends the model several calls, and most of each call repeats
an earlier one: the same system prompt, the same tool list, and the
conversation so far. A server with prefix caching reuses that work instead of
computing it again. Synthetic benchmark prompts share nothing, so they cannot
show it.

This notebook runs the 25 shopping journeys, reads every model call they made
from Phoenix, and writes them as a trace file that
[AIPerf](https://github.com/ai-dynamo/aiperf) can send to a model server again,
request for request. Notebook 5 replays it against a self-hosted NIM to
measure prefix-cache hits, time to first token, and throughput.

| The trace keeps | Taken from the call's span in Phoenix |
|---|---|
| The exact messages the call sent | `llm.input_messages` |
| The tool definitions, `tool_choice`, temperature, and thinking off | `llm.invocation_parameters` |
| The reply length, sent again as `max_tokens` | `llm.token_count.completion` |
| The conversation the call belongs to | `session.id` |
| The agent's own time between calls, running tools | span start and end times |

It leaves out how long the hosted model took, so a replay measures the server
under test, not the endpoint the trace was captured on.

### In this notebook

1. Confirm the stack is ready
2. Run the journeys
3. Build the trace
4. Check it is complete
5. Read the trace
6. See what a prefix cache can reuse
7. Save it
8. Validate it with AIPerf

### Before you start

- **Notebooks 1 to 3 done**, with the deployment still running.
- **Tracing on**, the default: the agent exports spans to Phoenix. Step 1
  checks.
- **Time:** about 35 minutes, almost all of it the journeys running.
- **Which model:** the trace records the requests that *this* model's replies
  led to. A different model picks other tools and writes other replies, so the
  calls that follow differ. Capture on the model you will test when you can,
  and never compare replays of traces captured on different models.

## Step 1: Confirm the Stack Is Ready

**Run** the next cell. It checks that the agent and Phoenix respond, and shows
where the agent sends its spans and which model it calls.

In [ ]:
from helpers import CHAIN_SERVER, PHOENIX, status

print(f"Agent:   {CHAIN_SERVER}   status: {status(CHAIN_SERVER + '/health')}")
print(f"Phoenix: {PHOENIX}   status: {status(PHOENIX)}")
!echo "OTEL_EXPORTER_OTLP_ENDPOINT=$(docker exec chain-server printenv OTEL_EXPORTER_OTLP_ENDPOINT)"
!echo "LLM_MODEL=$(docker exec chain-server printenv LLM_MODEL)"

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
Agent:   http://localhost:8009   status: 200
Phoenix: http://localhost:6006   status: 200
OTEL_EXPORTER_OTLP_ENDPOINT=http://otel-collector:4318
LLM_MODEL=nvidia/nvidia/nemotron-3.5-super-text-preview
</pre>

**If it does not match:**

| Symptom | Action |
|---|---|
| Agent status is not `200` | Confirm the deployment is running (Notebook 1, Step 5). |
| Phoenix status is not `200` | Start it: `docker compose up -d phoenix otel-collector`. |
| `OTEL_EXPORTER_OTLP_ENDPOINT` is empty | Tracing is off, so there is nothing to capture. Unset `OTEL_SDK_DISABLED` and restart the agent service: `docker compose up -d chain-server`. |

## Step 2: Run the Journeys

The replay runner from Notebook 3 sends each journey's shopper messages word
for word. Every model call the agent makes along the way lands in Phoenix,
under the conversation's id.

**Run** the next cell. It notes the time, then runs the 25 journeys, one at a
time. The hosted endpoint limits how many requests it accepts at once; at 4
conversations it starts refusing calls within minutes, and a refused call
changes the conversation that follows. On your own NIM, raise `CONCURRENCY`:
the trace keeps each conversation's calls in order, so running them side by
side changes nothing in it, only how long you wait.

To try the notebook quickly first, set `JOURNEYS = ["J01", "J21"]`.

In [ ]:
import sys

from helpers import new_conversation
from trace_capture import journey_ids, now

JOURNEYS = journey_ids()
CONCURRENCY = 1

label = new_conversation("trace")
since = now()
print(f"Capture {label}: {len(JOURNEYS)} journeys, {CONCURRENCY} at once, from {since}")
!cd .. && {sys.executable} -m tests.evaluation.src.replay --only {",".join(JOURNEYS)} --label {label} --concurrency {CONCURRENCY}

**You should see** one line per journey as it finishes, then the totals:

<pre style="font-size: 0.8em; line-height: 1.35;">
Capture trace-02401c23: 25 journeys, 1 at once, from 2026-09-27T21:09:29.766287+00:00
  assistant healthy at http://localhost:8009
  25 runs, one at a time, label trace-02401c23
  ok   J01_wedding_abroad
  ok   J02_video_look_full
  ...
  ok   J25_a_size_before_a_product

  pass 22  fail 3  error 0
</pre>

A journey that fails a check or two still belongs in the trace. What the model
is asked to do is the load, whether or not the answer passed.

**If it does not match:**

| Symptom | Action |
|---|---|
| Most journeys fail, with `products_min: 0 products shown` from the first turn | The endpoint is refusing requests. Confirm with `docker logs chain-server 2>&1 \| grep -c "429 Too Many Requests"`, wait a few minutes, and run the cell again with a lower `CONCURRENCY`. |

## Step 3: Build the Trace

Each model call is one span in Phoenix, named `ChatOpenAI`. Its attributes hold
the whole request, flattened: `llm.input_messages.3.message.role`,
`llm.input_messages.3.message.tool_calls.0.tool_call.function.name`, and so on.
`trace_capture.py` turns each one back into the request it was, as one line of
AIPerf's `mooncake_trace` format.

**Run** the next cell. It reads the capture's calls from Phoenix, waiting for
the last ones to arrive, and builds the trace.

In [ ]:
from helpers import REPO
from trace_capture import capture

RESULTS = REPO / "tests" / "evaluation" / "results" / "val" / label

lines, manifest, checks, refused = capture(label, since, RESULTS)
print(f"{len(lines)} model calls from {len(checks)} conversations, {refused} refused by the endpoint")

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
890 model calls from 25 conversations, 0 refused by the endpoint
</pre>

A refused call never reached the model, so it stays out of the trace. Any
refusal also means the conversation after it went differently, so the capture
is not the journeys as they run: capture again with a lower `CONCURRENCY`.

Reading the calls takes a minute or two. Phoenix can filter spans by name,
conversation, or time itself, but on a project with months of spans each
filtered query takes minutes, so `capture` pages back through the newest spans
until it reaches the start of the run.

## Step 4: Check It Is Complete

The replay runner kept its own count of model calls and prompt tokens for every
turn, from the agent's replies. A trace that matches it turn for turn holds
every call, in the right turn, with the prompt the model actually read.

**Run** the next cell.

In [ ]:
for check in checks:
    marks = "calls ok " if check["calls_match"] else "CALLS MISSING"
    marks += "  tokens ok" if check["prompt_tokens_match"] else "  TOKENS DIFFER"
    print(f"{check['scenario']:<34} {check['turns']:>3} turns {check['calls']:>4} calls   {marks}   ({check['outcome']})")

complete = not refused and all(check["calls_match"] and check["prompt_tokens_match"] for check in checks)
print()
print("Complete: every call, in its turn, with its prompt" if complete else "Incomplete: see the table below")
if refused:
    print(f"{refused} calls were refused by the endpoint: capture again with a lower CONCURRENCY")

**You should see** every journey marked `calls ok` and `tokens ok`:

<pre style="font-size: 0.8em; line-height: 1.35;">
J01_wedding_abroad                  19 turns   82 calls   calls ok   tokens ok   (pass)
J02_video_look_full                 13 turns   58 calls   calls ok   tokens ok   (pass)
...
J25_a_size_before_a_product          2 turns    9 calls   calls ok   tokens ok   (pass)

Complete: every call, in its turn, with its prompt
</pre>

**If it does not match:**

| Symptom | Action |
|---|---|
| `CALLS MISSING` on the last journeys to finish | Phoenix had not received their spans yet. Run Step 3 again. |
| `CALLS MISSING` that stays | A span was dropped on the way. Run that journey again under a new label, with `JOURNEYS = ["J07"]`, and use that capture. |
| `TOKENS DIFFER` | The capture holds a different call than the one the replay counted. Do not use it: capture again. |

## Step 5: Read the Trace

Every turn makes three kinds of model call:

| Call | What it does | Tools offered |
|---|---|---|
| **Skill selection** | Picks the shopper skills for the turn. | Only `activate_shopper_skills_tool`, forced by `tool_choice` |
| **Agent** | Searches, checks, and edits the cart, one tool call at a time, then writes the reply. Runs two or more times. | The skills' shopping tools |
| **Response editor** | Rewrites the reply against the facts the tools returned. | None |

**Run** the next cell. It counts each kind and shows how long their prompts are.

In [ ]:
from collections import defaultdict
from statistics import median, quantiles

from trace_capture import kind

prompts = defaultdict(list)
for line, row in zip(lines, manifest, strict=True):
    prompts[kind(line)].append(row["prompt_tokens"])

print(f"{'call':<18}{'calls':>7}{'median prompt':>15}{'p95':>9}{'longest':>9}   tokens")
for name in ("skill selection", "agent", "response editor"):
    tokens = prompts[name]
    p95 = quantiles(tokens, n=20, method="inclusive")[-1]
    print(f"{name:<18}{len(tokens):>7}{median(tokens):>15,.0f}{p95:>9,.0f}{max(tokens):>9,}")

turns = len({(row["session_id"], row["turn"]) for row in manifest})
print()
print(f"{len(lines)} calls over {turns} turns: {len(lines) / turns:.1f} per turn")
print(f"{sum(row['prompt_tokens'] for row in manifest):,} prompt tokens, "
      f"{sum(row['completion_tokens'] for row in manifest):,} completion tokens")

**You should see** agent calls carrying most of the prompt tokens. Counts
differ a little from run to run, as the agent's choices do:

<pre style="font-size: 0.8em; line-height: 1.35;">
call                calls  median prompt      p95  longest   tokens
skill selection       210          3,416    6,638    7,587
agent                 468         16,530   21,008   25,229
response editor       212          5,106    9,376   18,435

890 calls over 208 turns: 4.3 per turn
9,680,011 prompt tokens, 136,734 completion tokens
</pre>

**Run** the next cell to see one line of the trace, with long text cut short.

In [ ]:
import json

example = next(line for line in lines if kind(line) == "agent" and len(line["messages"]) >= 6)

def cut(value, size=70):
    text = value if isinstance(value, str) else json.dumps(value)
    return text if len(text) <= size else text[:size] + "..."

print("session_id:   ", example["session_id"])
print("delay (ms):   ", example.get("delay"), "  the agent's time since the previous reply")
print("output_length:", example["output_length"])
print("extra:        ", example["extra"])
print("tools:        ", [tool["function"]["name"] for tool in example["tools"]])
print("messages:")
for message in example["messages"]:
    calls = [call["function"]["name"] for call in message.get("tool_calls", [])]
    print(f"  {message['role']:<10}", cut(message["content"]), f"-> calls {calls}" if calls else "")

**You should see** a request as the agent sent it: the system prompt, the
shopper's message, and each tool call with its result:

<pre style="font-size: 0.8em; line-height: 1.35;">
session_id:    trace-02401c23-J01_wedding_abroad-0
delay (ms):    2867.5   the agent's time since the previous reply
output_length: 768
extra:         {'temperature': 0.0, 'parallel_tool_calls': False, 'chat_template_kwargs': {'enable_thinking': False}}
tools:         ['search_catalog_tool', 'get_product_details_tool', 'resolve_conversation_products_tool', ...]
messages:
  system     [{"type": "text", "text": "You are a retail shopping assistant for the...
  user       REQUEST ID: d1f509d1-19a2-4d7c-a791-03ca15970bb5 ...
  assistant   -> calls ['activate_shopper_skills_tool']
  tool       SHOPPER_SKILL_ACTIVATION_COMPLETE: /shopper/outfit-styling/SKILL.md
  assistant   -> calls ['search_catalog_tool']
  tool       SCOPE 1 (dresses): ...
</pre>

- **`extra`** is merged into the request as it is sent: greedy decoding,
  thinking off, one tool call at a time, and here the tool the model must call.
- **`output_length`** becomes `max_tokens`. Notebook 5 also sets `ignore_eos`,
  so every reply is exactly as long as the one recorded.

## Step 6: See What a Prefix Cache Can Reuse

A prefix cache keeps the computed start of each request. When a later request
begins with the same text, the server skips that part and computes only the
rest. For each call, the next cell measures the longest start it shares with
any earlier call:

- **Within the conversation:** the conversation so far, and the prompts the
  turn before it used.
- **Across conversations:** the system prompts and tool lists every
  conversation shares.

**Run** the next cell.

In [ ]:
from trace_capture import prefix_reuse

reuse = prefix_reuse(lines)

def weighted(rows, field):
    return sum(row[field] * row["chars"] for row in rows) / sum(row["chars"] for row in rows)

by_kind = defaultdict(list)
for line, row in zip(lines, reuse, strict=True):
    by_kind[kind(line)].append(row)

print(f"{'call':<18}{'within the conversation':>25}{'across conversations':>23}")
for name in ("skill selection", "agent", "response editor", "all calls"):
    rows = reuse if name == "all calls" else by_kind[name]
    print(f"{name:<18}{weighted(rows, 'from_session'):>25.0%}{weighted(rows, 'from_any'):>23.0%}")

**You should see** most of each agent call's request already sent by an
earlier call:

<pre style="font-size: 0.8em; line-height: 1.35;">
call                within the conversation   across conversations
skill selection                         41%                    47%
agent                                   77%                    86%
response editor                         43%                    50%
all calls                               71%                    79%
</pre>

Each share is weighted by request length, as the server's work is.

This is an estimate from the text, and a ceiling. The model's chat template
decides where the tool list goes, so a turn whose tools change can share less
than this. The server counts in tokens, and only reuses what it has not yet
evicted. Notebook 5 measures the real hit rate from the server's own counters;
this is the number to hold it against.

## Step 7: Save It

**Run** the next cell. It writes the capture to `traces/<label>/`:

| File | Holds |
|---|---|
| `journeys.jsonl` | The trace, one line per model call, for AIPerf. |
| `manifest.jsonl` | What was recorded for each call: tokens, model time, and its span in Phoenix. |
| `capture.json` | How the capture was made: the model, journeys, and time. |

The trace holds every prompt in full, carts included. `traces/` is in
`.gitignore`, so it stays out of the repository.

In [ ]:
from trace_capture import TRACES, write_jsonl

folder = TRACES / label
write_jsonl(folder / "journeys.jsonl", lines)
write_jsonl(folder / "manifest.jsonl", manifest)
(folder / "capture.json").write_text(json.dumps({
    "label": label,
    "since": since,
    "model": manifest[0]["model"],
    "journeys": JOURNEYS,
    "concurrency": CONCURRENCY,
    "calls": len(lines),
    "complete": complete,
    "format": "aiperf mooncake_trace",
}, indent=2) + "\n")

for path in sorted(folder.iterdir()):
    print(f"{path.relative_to(REPO)}   {path.stat().st_size / 1e6:.1f} MB")

**You should see** the three files:

<pre style="font-size: 0.8em; line-height: 1.35;">
traces/trace-02401c23/capture.json   0.0 MB
traces/trace-02401c23/journeys.jsonl   37.6 MB
traces/trace-02401c23/manifest.jsonl   0.3 MB
</pre>

## Step 8: Validate It with AIPerf

AIPerf checks every line of a trace when it loads one, so a mistake would
otherwise first show up on the GPU host. The next cell runs that same check
here, with AIPerf's own schema.

AIPerf goes in a virtual environment of its own, because it pins libraries the
notebooks also use. **Install** it once, in a terminal:

```bash
python3 -m venv ~/aiperf && ~/aiperf/bin/pip install aiperf
```

**Run** the next cell.

In [ ]:
import subprocess
from pathlib import Path

AIPERF_PYTHON = Path.home() / "aiperf" / "bin" / "python"
VALIDATE = """
import sys
from aiperf.dataset.loader.models import MooncakeTrace
lines = open(sys.argv[1]).read().splitlines()
for line in lines:
    MooncakeTrace.model_validate_json(line)
print(f"AIPerf accepts all {len(lines)} lines")
"""

_ = subprocess.run([AIPERF_PYTHON, "-c", VALIDATE, folder / "journeys.jsonl"], check=True)

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
AIPerf accepts all 890 lines
</pre>

**If it does not match:**

| Symptom | Action |
|---|---|
| `FileNotFoundError` | Install AIPerf, above. |
| `ValidationError` | Note the line number and the field it names, and report it with the trace. |

## What Notebook 5 Does with It

Notebook 5 sends this trace to a NIM on its own GPU host, first one
conversation at a time, then many at once, with prefix caching on and off. At
its core is one AIPerf command:

```bash
aiperf profile \
  --model <the NIM's model> --url http://<gpu-host>:8000 \
  --endpoint-type chat --streaming \
  --input-file traces/<label>/journeys.jsonl --custom-dataset-type mooncake_trace \
  --no-fixed-schedule --concurrency 16 \
  --extra-inputs ignore_eos:true
```

Copy `traces/<label>/` to that host, or run Notebook 5 from this one.